Now I create clean, standardized, and relationally consistent data that analysts can safely query.
- deduplication: Remove duplicate Accounts or Contacts with same ID
- Type casting / normalization: Convert text dates to proper timestamps, format phone numbers
- Relationship validation: Ensure every Contact’s AccountId exists in Account table
- Filtering / partitioning: Drop test/demo data, keep only active Accounts
- Incremental logic: Use LastModifiedDate to process only new/updated records

In [0]:
CREATE SCHEMA IF NOT EXISTS hive_metastore.silver_salesforce; 
CREATE OR REPLACE TABLE hive_metastore.silver_salesforce.account AS
SELECT
  Id,
  Name,
  Type,
  Industry,
  BillingCountry,
  OwnerId,
  CAST(CreatedDate AS TIMESTAMP) AS CreatedDate,
  CAST(LastModifiedDate AS TIMESTAMP) AS LastModifiedDate
FROM hive_metastore.bronze_salesforce.account
WHERE Id IS NOT NULL
  AND Name IS NOT NULL;

-- now the same operations runs for the remaining 3 tables
-- CONTACT ➜ Silver
CREATE OR REPLACE TABLE hive_metastore.silver_salesforce.contact AS
SELECT
  Id,
  AccountId,
  FirstName,
  LastName,
  Email,
  Phone,
  CAST(CreatedDate      AS TIMESTAMP) AS CreatedDate,
  CAST(LastModifiedDate AS TIMESTAMP) AS LastModifiedDate
FROM hive_metastore.bronze_salesforce.contact
WHERE Id IS NOT NULL
  -- keep contacts with at least one name populated
  AND (FirstName IS NOT NULL OR LastName IS NOT NULL);

-- TASK ➜ Silver
CREATE OR REPLACE TABLE hive_metastore.silver_salesforce.task AS
SELECT
  Id,
  WhatId,        -- usually Account/Opportunity/etc.
  WhoId,         -- usually Contact/Lead
  Subject,
  Status,
  Priority,
  CAST(ActivityDate     AS DATE)      AS ActivityDate,      -- task due date
  CAST(CreatedDate      AS TIMESTAMP) AS CreatedDate,
  CAST(LastModifiedDate AS TIMESTAMP) AS LastModifiedDate
FROM hive_metastore.bronze_salesforce.task
WHERE Id IS NOT NULL
  AND Subject IS NOT NULL;

-- CASE ➜ Silver
CREATE OR REPLACE TABLE hive_metastore.silver_salesforce.case AS
SELECT
  Id,
  AccountId,
  ContactId,
  CaseNumber,
  Subject,
  Status,
  Priority,
  Origin,
  CAST(CreatedDate      AS TIMESTAMP) AS CreatedDate,
  CAST(LastModifiedDate AS TIMESTAMP) AS LastModifiedDate,
  -- handle blanks gracefully
  CAST(NULLIF(ClosedDate, '') AS TIMESTAMP) AS ClosedDate
FROM hive_metastore.bronze_salesforce.case
WHERE Id IS NOT NULL
  AND CaseNumber IS NOT NULL;
